# Features and the Three Classifiers


### Load the Processed Tables


In [1]:
from pathlib import Path

import pandas as pd

seed = 42

candidate_paths = [
    Path("data/processed"),
    Path("../data/processed"),
]
processed_dir = next((path for path in candidate_paths if path.exists()), None)
if processed_dir is None:
    raise FileNotFoundError(
        "Could not find data/processed. Run Objective_1_Tan_Dai_Ngo.ipynb first."
    )
for required in ["lyrics_clean.csv", "splits.csv", "tags_normalised.csv",
                 "shared_vocabulary_50.csv"]:
    if not (processed_dir / required).exists():
        raise FileNotFoundError(
            f"Could not find {required}. Run Objective_1_Tan_Dai_Ngo.ipynb first."
        )
splits = pd.read_csv(processed_dir / "splits.csv", dtype={"id": str})
tags_frame = pd.read_csv(processed_dir / "tags_normalised.csv", dtype={"id": str})
vocabulary_frame = pd.read_csv(processed_dir / "shared_vocabulary_50.csv")

tag_vocabulary = list(vocabulary_frame["tag"])

print(f"Splits shape: {splits.shape}")
print(f"Tags shape: {tags_frame.shape}")
print(f"Tags in the shared vocabulary: {len(tag_vocabulary)}")
print("Same ids in both tables:", set(splits["id"]) == set(tags_frame["id"]))
splits.head()

Splits shape: (82810, 4)
Tags shape: (82810, 2)
Tags in the shared vocabulary: 50
Same ids in both tables: True


,id,artist,random_split,artist_split
0,0009fFIM1eYThaPg,Cheryl,test,train
1,0010xmHR6UICBOYT,Oddisee,train,train
2,002Jyd0vN4HyCpqL,Blue Öyster Cult,train,test
3,006TYKNjNxWjfKjy,Rhapsody,train,valid
4,007LIJOPQ4Sb98qV,The Chameleons,valid,train


### Read the Lyrics Back


In [2]:
import hashlib
import re

candidate_paths = [
    Path("music4all-en"),
    Path("../music4all-en"),
]
data_dir = next((path for path in candidate_paths if path.exists()), None)
if data_dir is None:
    raise FileNotFoundError(
        "Could not find music4all-en. Unzip the subset to the repo root."
    )
lyrics_dir = data_dir / "lyrics"
marker_pattern = re.compile(r"^\s*\[[^\]]{1,30}\]\s*$")


def clean_lines(text):
    kept = []
    for line in text.replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        line = line.rstrip()
        if not line.strip():
            continue
        if marker_pattern.match(line):
            continue
        kept.append(line)

    return "\n".join(kept)


def flatten(text):
    return re.sub(r"\s+", " ", clean_lines(text).lower()).strip()


track_ids = list(splits["id"])
documents = []
for track_id in track_ids:
    path = lyrics_dir / (track_id + ".txt")
    raw = path.read_text(encoding="utf-8", errors="replace")
    documents.append(flatten(raw))

word_counts = pd.Series([len(text.split()) for text in documents], index=track_ids)
fingerprints = pd.Series(
    [hashlib.sha1(text.encode("utf-8")).hexdigest() for text in documents],
    index=track_ids,
)
expected = pd.read_csv(processed_dir / "lyrics_clean.csv", dtype={"id": str}).set_index("id")

print(f"Documents read: {len(documents)}")
print(f"Empty documents: {int((word_counts == 0).sum())}")
print(f"Mean words: {round(word_counts.mean(), 1)}")
print(f"Median words: {int(word_counts.median())}")
print("Fingerprints matching lyrics_clean.csv:",
      int((fingerprints == expected.loc[track_ids, "fingerprint"]).sum()), "of", len(track_ids))
print("Word counts matching lyrics_clean.csv:",
      int((word_counts == expected.loc[track_ids, "n_words"]).sum()), "of", len(track_ids))

Documents read: 82810
Empty documents: 0
Mean words: 242.9
Median words: 223
Fingerprints matching lyrics_clean.csv: 82810 of 82810
Word counts matching lyrics_clean.csv: 82810 of 82810


### Build the Target Matrix


In [3]:
import numpy as np

tag_lists = {}
for record in tags_frame.itertuples(index=False):
    if isinstance(record.tags, str) and record.tags != "":
        tag_lists[record.id] = record.tags.split(",")
    else:
        tag_lists[record.id] = []

position = {}
for i in range(len(tag_vocabulary)):
    position[tag_vocabulary[i]] = i

target = np.zeros((len(track_ids), len(tag_vocabulary)), dtype=np.int8)
row = 0
for track_id in track_ids:
    for tag in tag_lists[track_id]:
        if tag in position:
            target[row, position[tag]] = 1
    row = row + 1

cardinality = target.sum(axis=1).mean()

print(f"Target shape: {target.shape}")
print(f"Cardinality: {round(float(cardinality), 2)}")
print(f"Density: {round(float(cardinality) / len(tag_vocabulary), 4)}")
print(f"Rows with no tag: {int((target.sum(axis=1) == 0).sum())}")
print(f"Smallest support over all rows: {int(target.sum(axis=0).min())}")
print(f"Largest support over all rows: {int(target.sum(axis=0).max())}")

Target shape: (82810, 50)
Cardinality: 2.46
Density: 0.0493
Rows with no tag: 5092
Smallest support over all rows: 1175
Largest support over all rows: 20727


### Index the Two Splits


In [4]:
split_rows = {}
for name in ["random", "artist"]:
    column = splits[name + "_split"]
    parts = {}
    for part in ["train", "valid", "test"]:
        parts[part] = np.flatnonzero(column == part)
    split_rows[name] = parts

    print(name, "split:",
          "train", len(parts["train"]),
          "valid", len(parts["valid"]),
          "test", len(parts["test"]))
documents = np.array(documents, dtype=object)

print(f"Documents ready to index by row: {documents.shape[0]}")

random split: train 57966 valid 12421 test 12423
artist split: train 57997 valid 12424 test 12389
Documents ready to index by row: 82810


## (basic) bag-of-words plus Naive Bayes per tag, micro-F1 on validation


### Bag of Words Features


In [5]:
from sklearn.feature_extraction.text import CountVectorizer


features = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    vectoriser = CountVectorizer()
    X_train = vectoriser.fit_transform(documents[parts["train"]])
    X_valid = vectoriser.transform(documents[parts["valid"]])
    features[name] = {
        "vectoriser": vectoriser,
        "X_train": X_train,
        "X_valid": X_valid,
    }

    print(f"{name} split: terms {len(vectoriser.vocabulary_)} train nonzeros {X_train.nnz} train "
          f"density {round(X_train.nnz / (X_train.shape[0] * X_train.shape[1]), 6)}")

random split: terms 86194 train nonzeros 5049464 train density 0.001011


artist split: terms 85307 train nonzeros 5076907 train density 0.001026


### Naive Bayes per Tag


In [6]:
import time

from sklearn.naive_bayes import MultinomialNB

predictions = {}
fit_seconds = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    X_train = features[name]["X_train"]
    X_valid = features[name]["X_valid"]
    y_train = target[parts["train"]]
    predicted = np.zeros((X_valid.shape[0], len(tag_vocabulary)), dtype=np.int8)
    started = time.perf_counter()
    for j in range(len(tag_vocabulary)):
        model = MultinomialNB()
        model.fit(X_train, y_train[:, j])
        predicted[:, j] = model.predict(X_valid)

    elapsed = time.perf_counter() - started
    predictions[name] = predicted
    fit_seconds[name] = elapsed

    print(f"{name} split: models {len(tag_vocabulary)} seconds {round(elapsed, 1)} seconds per "
          f"tag {round(elapsed / len(tag_vocabulary), 2)}")

random split: models 50 seconds 1.1 seconds per tag 0.02


artist split: models 50 seconds 1.1 seconds per tag 0.02


### Micro F1 on Validation


In [7]:
from sklearn.metrics import precision_recall_fscore_support

summary_rows = []
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_valid = target[parts["valid"]]
    predicted = predictions[name]
    micro = precision_recall_fscore_support(
        y_valid, predicted, average="micro", zero_division=0,
    )
    macro = precision_recall_fscore_support(
        y_valid, predicted, average="macro", zero_division=0,
    )
    summary_rows.append({
        "split": name,
        "terms": len(features[name]["vectoriser"].vocabulary_),
        "micro_precision": round(micro[0], 3),
        "micro_recall": round(micro[1], 3),
        "micro_f1": round(micro[2], 3),
        "macro_f1": round(macro[2], 3),
        "true_positive": int(((y_valid == 1) & (predicted == 1)).sum()),
        "false_positive": int(((y_valid == 0) & (predicted == 1)).sum()),
        "false_negative": int(((y_valid == 1) & (predicted == 0)).sum()),
        "fit_seconds": round(fit_seconds[name], 1),
    })

    print(f"{name} split: micro precision {round(micro[0], 3)} micro recall {round(micro[1], 3)} "
          f"micro F1 {round(micro[2], 3)} macro F1 {round(macro[2], 3)}")
bow_summary = pd.DataFrame(summary_rows)

display(bow_summary)

random split: micro precision 0.233 micro recall 0.293 micro F1 0.26 macro F1 0.152
artist split: micro precision 0.221 micro recall 0.26 micro F1 0.239 macro F1 0.131


,split,terms,micro_precision,micro_recall,micro_f1,macro_f1,true_positive,false_positive,false_negative,fit_seconds
0,random,86194,0.233,0.293,0.260,0.152,8960,29442,21647,1.1
1,artist,85307,0.221,0.260,0.239,0.131,8026,28216,22856,1.1


### Per Tag Scores


In [8]:
bow_tag_tables = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_valid = target[parts["valid"]]
    predicted = predictions[name]
    precision, recall, f1, support = precision_recall_fscore_support(
        y_valid, predicted, average=None, zero_division=0,
    )
    bow_tag_tables[name] = pd.DataFrame({
        "tag": tag_vocabulary,
        "support": support,
        "predicted": predicted.sum(axis=0),
        "precision": precision.round(3),
        "recall": recall.round(3),
        "f1": f1.round(3),
    })
table = bow_tag_tables["random"].sort_values("f1", ascending=False)

print("Best five tags on the random split:")
display(table.head(5))
print("Worst five tags on the random split:")
display(table.tail(5))

Best five tags on the random split:


,tag,support,predicted,precision,recall,f1
16,hip-hop,594,894,0.465,0.700,0.559
19,rap,455,763,0.444,0.745,0.557
1,pop,2362,4117,0.368,0.642,0.468
0,rock,3161,4613,0.368,0.537,0.437
38,death metal,273,671,0.265,0.652,0.377


Worst five tags on the random split:


,tag,support,predicted,precision,recall,f1
44,pop rock,200,107,0.019,0.010,0.013
40,dream pop,226,101,0.020,0.009,0.012
46,fip,189,141,0.014,0.011,0.012
48,shoegaze,189,83,0.012,0.005,0.007
45,lo-fi,235,116,0.009,0.004,0.006


### Save the Baseline Tables


In [9]:
for name in ["random", "artist"]:
    out_path = processed_dir / ("nb_bow_per_tag_" + name + ".csv")
    bow_tag_tables[name].to_csv(out_path, index=False)

    print(f"Wrote: {out_path}")

bow_summary.to_csv(processed_dir / "nb_bow_summary.csv", index=False)

print("Wrote:", processed_dir / "nb_bow_summary.csv")
bow_summary

Wrote: data\processed\nb_bow_per_tag_random.csv
Wrote: data\processed\nb_bow_per_tag_artist.csv
Wrote: data\processed\nb_bow_summary.csv


,split,terms,micro_precision,micro_recall,micro_f1,macro_f1,true_positive,false_positive,false_negative,fit_seconds
0,random,86194,0.233,0.293,0.260,0.152,8960,29442,21647,1.1
1,artist,85307,0.221,0.260,0.239,0.131,8026,28216,22856,1.1


## (basic) switch to TF-IDF, sweep min_df, max_df and n-gram range


### The Sweep Grid


In [10]:
import itertools

min_df_values = [1, 2, 5, 10, 20]
max_df_values = [0.5, 0.9, 1.0]
ngram_values = [(1, 1), (1, 2)]
threshold_values = [0.5, 0.3, 0.2, 0.15, 0.1, 0.05, 0.02]

configurations = []
for ngram_range, min_df, max_df in itertools.product(ngram_values, min_df_values, max_df_values):
    configurations.append({"ngram_range": ngram_range, "min_df": min_df, "max_df": max_df})

print(f"Configurations per split: {len(configurations)}")
print(f"Thresholds per configuration: {len(threshold_values)}")
print(f"Rows the sweep will produce: "
      f"{len(configurations) * len(threshold_values) * len(split_rows)}")
print(f"Naive Bayes models to train: {len(configurations) * len(split_rows) * len(tag_vocabulary)}")

Configurations per split: 30
Thresholds per configuration: 7
Rows the sweep will produce: 420
Naive Bayes models to train: 3000


### Run the Sweep


In [11]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


sweep_rows = []
mass_rows = []
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_train = target[parts["train"]]
    y_valid = target[parts["valid"]]
    counts = CountVectorizer()
    counts_mass = float(counts.fit_transform(documents[parts["train"]]).sum(axis=1).mean())
    for configuration in configurations:
        started = time.perf_counter()
        vectoriser = TfidfVectorizer(
            ngram_range=configuration["ngram_range"],
            min_df=configuration["min_df"],
            max_df=configuration["max_df"],
        )
        X_train = vectoriser.fit_transform(documents[parts["train"]])
        X_valid = vectoriser.transform(documents[parts["valid"]])
        scores = np.zeros((X_valid.shape[0], len(tag_vocabulary)))
        for j in range(len(tag_vocabulary)):
            model = MultinomialNB()
            model.fit(X_train, y_train[:, j])
            scores[:, j] = model.predict_proba(X_valid)[:, 1]

        elapsed = time.perf_counter() - started
        mass_rows.append({
            "split": name,
            "ngram_range": str(configuration["ngram_range"]),
            "min_df": configuration["min_df"],
            "max_df": configuration["max_df"],
            "terms": len(vectoriser.vocabulary_),
            "counts_mass_per_document": round(counts_mass, 1),
            "tfidf_mass_per_document": round(float(X_train.sum(axis=1).mean()), 2),
        })
        for threshold in threshold_values:
            predicted = (scores >= threshold).astype(np.int8)
            micro = precision_recall_fscore_support(
                y_valid, predicted, average="micro", zero_division=0,
            )
            macro = precision_recall_fscore_support(
                y_valid, predicted, average="macro", zero_division=0,
            )
            sweep_rows.append({
                "split": name,
                "ngram_range": str(configuration["ngram_range"]),
                "min_df": configuration["min_df"],
                "max_df": configuration["max_df"],
                "threshold": threshold,
                "terms": len(vectoriser.vocabulary_),
                "micro_precision": round(micro[0], 3),
                "micro_recall": round(micro[1], 3),
                "micro_f1": round(micro[2], 3),
                "macro_f1": round(macro[2], 3),
                "predicted": int(predicted.sum()),
            })
        best_here = max(
            [r for r in sweep_rows
             if r["split"] == name
             and r["ngram_range"] == str(configuration["ngram_range"])
             and r["min_df"] == configuration["min_df"]
             and r["max_df"] == configuration["max_df"]],
            key=lambda r: r["micro_f1"],
        )

        print(name,
              "ngram", configuration["ngram_range"],
              "min_df", configuration["min_df"],
              "max_df", configuration["max_df"],
              "terms", len(vectoriser.vocabulary_),
              "best micro F1", best_here["micro_f1"],
              "at threshold", best_here["threshold"],
              "seconds", round(elapsed, 1))
sweep = pd.DataFrame(sweep_rows)
mass = pd.DataFrame(mass_rows)

print()
print(f"Sweep finished, rows: {len(sweep)}")

random ngram (1, 1) min_df 1 max_df 0.5 terms 86175 best micro F1 0.216 at threshold 0.02 seconds 6.6


random ngram (1, 1) min_df 1 max_df 0.9 terms 86193 best micro F1 0.209 at threshold 0.02 seconds 6.7


random ngram (1, 1) min_df 1 max_df 1.0 terms 86194 best micro F1 0.208 at threshold 0.02 seconds 6.7


random ngram (1, 1) min_df 2 max_df 0.5 terms 42000 best micro F1 0.236 at threshold 0.02 seconds 6.5


random ngram (1, 1) min_df 2 max_df 0.9 terms 42018 best micro F1 0.236 at threshold 0.02 seconds 6.4


random ngram (1, 1) min_df 2 max_df 1.0 terms 42019 best micro F1 0.236 at threshold 0.02 seconds 6.4


... (50 outputs left out)

artist ngram (1, 2) min_df 10 max_df 1.0 terms 100106 best micro F1 0.252 at threshold 0.02 seconds 15.5


artist ngram (1, 2) min_df 20 max_df 0.5 terms 54169 best micro F1 0.267 at threshold 0.05 seconds 14.7


artist ngram (1, 2) min_df 20 max_df 0.9 terms 54187 best micro F1 0.266 at threshold 0.05 seconds 14.8


artist ngram (1, 2) min_df 20 max_df 1.0 terms 54188 best micro F1 0.266 at threshold 0.05 seconds 14.6

Sweep finished, rows: 420


### The Feature Mass Gap


In [12]:
summary_mass = mass.groupby("split")[
    ["counts_mass_per_document", "tfidf_mass_per_document"]
].agg(["mean", "min", "max"]).round(2)

print("Mass per document, counts against TF-IDF:")
display(summary_mass)

print("Ratio at the lightest and heaviest TF-IDF configuration:")
for name in ["random", "artist"]:
    rows = mass[mass["split"] == name]
    print(" ", name,
          "counts", rows["counts_mass_per_document"].iloc[0],
          "TF-IDF from", rows["tfidf_mass_per_document"].min(),
          "to", rows["tfidf_mass_per_document"].max(),
          "so a factor of",
          round(rows["counts_mass_per_document"].iloc[0] / rows["tfidf_mass_per_document"].max(), 1),
          "at best")

Mass per document, counts against TF-IDF:


counts_mass_per_document               tfidf_mass_per_document        \
                           mean    min    max                    mean   min   
split                                                                         
artist                    231.8  231.8  231.8                    8.02  5.91   
random                    230.0  230.0  230.0                    8.01  5.90   

               
          max  
split          
artist  10.68  
random  10.67

Ratio at the lightest and heaviest TF-IDF configuration:
  random counts 230.0 TF-IDF from 5.9 to 10.67 so a factor of 21.6 at best
  artist counts 231.8 TF-IDF from 5.91 to 10.68 so a factor of 21.7 at best


### What the Default Cut Would Have Concluded


In [13]:
for name in ["random", "artist"]:
    at_default = sweep[(sweep["split"] == name) & (sweep["threshold"] == 0.5)]
    at_default = at_default.sort_values("micro_f1", ascending=False)
    jointly = sweep[sweep["split"] == name].sort_values("micro_f1", ascending=False)
    naive = at_default.iloc[0]
    honest = jointly.iloc[0]

    print(f"{name} split")
    print("  judged at 0.5 only : ngram", naive["ngram_range"],
          "min_df", naive["min_df"], "max_df", naive["max_df"],
          "micro F1", naive["micro_f1"], "macro F1", naive["macro_f1"])
    print("  judged jointly     : ngram", honest["ngram_range"],
          "min_df", honest["min_df"], "max_df", honest["max_df"],
          "threshold", honest["threshold"],
          "micro F1", honest["micro_f1"], "macro F1", honest["macro_f1"])

random split
  judged at 0.5 only : ngram (1, 1) min_df 20 max_df 0.5 micro F1 0.044 macro F1 0.036
  judged jointly     : ngram (1, 2) min_df 20 max_df 0.5 threshold 0.05 micro F1 0.273 macro F1 0.084
artist split
  judged at 0.5 only : ngram (1, 1) min_df 20 max_df 0.5 micro F1 0.036 macro F1 0.034
  judged jointly     : ngram (1, 2) min_df 20 max_df 0.5 threshold 0.05 micro F1 0.267 macro F1 0.079


### The Grid, Split by Split


In [14]:
best_per_configuration = {}
for name in ["random", "artist"]:
    rows = sweep[sweep["split"] == name]
    index = rows.groupby(["ngram_range", "min_df", "max_df"])["micro_f1"].idxmax()
    table = rows.loc[index].sort_values("micro_f1", ascending=False)
    best_per_configuration[name] = table

    print(f"{name} split, best five configurations at their best threshold:")
    display(table.head(5))
    print(f"{name} split, worst three:")
    display(table.tail(3))

random split, best five configurations at their best threshold:


,split,ngram_range,min_df,max_df,threshold,terms,micro_precision,micro_recall,micro_f1,macro_f1,predicted
194,random,"(1, 2)",20,0.5,0.05,53828,0.231,0.334,0.273,0.084,44235
87,random,"(1, 1)",20,0.5,0.15,10143,0.262,0.282,0.272,0.094,32939
201,random,"(1, 2)",20,0.9,0.05,53846,0.234,0.325,0.272,0.080,42464
208,random,"(1, 2)",20,1.0,0.05,53847,0.235,0.323,0.272,0.080,42086
101,random,"(1, 1)",20,1.0,0.15,10162,0.262,0.277,0.269,0.091,32258


random split, worst three:


,split,ngram_range,min_df,max_df,threshold,terms,micro_precision,micro_recall,micro_f1,macro_f1,predicted
111,random,"(1, 2)",1,0.5,0.02,1636816,0.258,0.008,0.015,0.003,890
118,random,"(1, 2)",1,0.9,0.02,1636834,0.250,0.006,0.012,0.003,752
125,random,"(1, 2)",1,1.0,0.02,1636835,0.249,0.006,0.012,0.003,734


artist split, best five configurations at their best threshold:


,split,ngram_range,min_df,max_df,threshold,terms,micro_precision,micro_recall,micro_f1,macro_f1,predicted
404,artist,"(1, 2)",20,0.5,0.05,54169,0.225,0.327,0.267,0.079,44775
411,artist,"(1, 2)",20,0.9,0.05,54187,0.229,0.318,0.266,0.076,43004
418,artist,"(1, 2)",20,1.0,0.05,54188,0.229,0.317,0.266,0.075,42695
297,artist,"(1, 1)",20,0.5,0.15,10254,0.258,0.271,0.264,0.085,32381
311,artist,"(1, 1)",20,1.0,0.15,10273,0.260,0.266,0.263,0.082,31633


artist split, worst three:


,split,ngram_range,min_df,max_df,threshold,terms,micro_precision,micro_recall,micro_f1,macro_f1,predicted
321,artist,"(1, 2)",1,0.5,0.02,1639452,0.252,0.007,0.013,0.003,806
328,artist,"(1, 2)",1,0.9,0.02,1639470,0.239,0.005,0.010,0.002,682
335,artist,"(1, 2)",1,1.0,0.02,1639471,0.239,0.005,0.010,0.002,661


### What Each Parameter Does on Its Own


In [15]:
for parameter in ["ngram_range", "min_df", "max_df", "threshold"]:
    if parameter == "threshold":
        grouped = sweep.groupby(["split", parameter])["micro_f1"].agg(["mean", "max"])
    else:
        frames = []
        for name in ["random", "artist"]:
            frames.append(best_per_configuration[name])
        combined = pd.concat(frames)
        grouped = combined.groupby(["split", parameter])["micro_f1"].agg(["mean", "max", "min"])

    print(f"Effect of {parameter}")
    display(grouped.round(3))

Effect of ngram_range


mean    max    min
split  ngram_range                     
artist (1, 1)       0.242  0.264  0.215
       (1, 2)       0.159  0.267  0.010
random (1, 1)       0.244  0.272  0.208
       (1, 2)       0.163  0.273  0.012

Effect of min_df


mean    max    min
split  min_df                     
artist 1       0.114  0.221  0.010
       2       0.143  0.236  0.048
       5       0.226  0.242  0.208
       10      0.252  0.254  0.252
       20      0.265  0.267  0.263
random 1       0.112  0.216  0.012
       2       0.147  0.236  0.055
       5       0.229  0.245  0.211
       10      0.258  0.260  0.257
       20      0.271  0.273  0.269

Effect of max_df


mean    max    min
split  max_df                     
artist 0.5     0.203  0.267  0.013
       0.9     0.199  0.266  0.010
       1.0     0.199  0.266  0.010
random 0.5     0.206  0.273  0.015
       0.9     0.202  0.272  0.012
       1.0     0.202  0.272  0.012

Effect of threshold


mean    max
split  threshold              
artist 0.02       0.183  0.253
       0.05       0.172  0.267
       0.10       0.145  0.259
       0.15       0.117  0.264
       0.20       0.086  0.241
       0.30       0.037  0.116
       0.50       0.011  0.036
random 0.02       0.187  0.259
       0.05       0.175  0.273
       0.10       0.149  0.268
       0.15       0.121  0.272
       0.20       0.092  0.247
       0.30       0.042  0.131
       0.50       0.015  0.044

### Compare Against the Bag of Words Baseline


In [16]:
comparison_rows = []
for name in ["random", "artist"]:
    reference = bow_summary[bow_summary["split"] == name].iloc[0]
    at_default = sweep[(sweep["split"] == name) & (sweep["threshold"] == 0.5)]
    naive = at_default.sort_values("micro_f1", ascending=False).iloc[0]
    best = sweep[sweep["split"] == name].sort_values("micro_f1", ascending=False).iloc[0]
    comparison_rows.append({
        "split": name,
        "bow_micro_f1": reference["micro_f1"],
        "tfidf_micro_f1_default_cut": naive["micro_f1"],
        "tfidf_micro_f1_tuned_cut": best["micro_f1"],
        "micro_gain": round(best["micro_f1"] - reference["micro_f1"], 3),
        "bow_macro_f1": reference["macro_f1"],
        "tfidf_macro_f1_tuned_cut": best["macro_f1"],
        "macro_gain": round(best["macro_f1"] - reference["macro_f1"], 3),
        "best_threshold": best["threshold"],
        "best_ngram_range": best["ngram_range"],
        "best_min_df": best["min_df"],
        "best_max_df": best["max_df"],
        "bow_terms": int(reference["terms"]),
        "tfidf_terms": int(best["terms"]),
    })

    print(name, "split:",
          "bag of words", reference["micro_f1"],
          "TF-IDF at the default cut", naive["micro_f1"],
          "TF-IDF at a cut of", best["threshold"], "gives", best["micro_f1"])
tfidf_comparison = pd.DataFrame(comparison_rows)

display(tfidf_comparison)

random split: bag of words 0.26 TF-IDF at the default cut 0.044 TF-IDF at a cut of 0.05 gives 0.273
artist split: bag of words 0.239 TF-IDF at the default cut 0.036 TF-IDF at a cut of 0.05 gives 0.267


,split,bow_micro_f1,tfidf_micro_f1_default_cut,tfidf_micro_f1_tuned_cut,micro_gain,bow_macro_f1,tfidf_macro_f1_tuned_cut,macro_gain,best_threshold,best_ngram_range,best_min_df,best_max_df,bow_terms,tfidf_terms
0,random,0.260,0.044,0.273,0.013,0.152,0.084,-0.068,0.05,"(1, 2)",20,0.5,86194,53828
1,artist,0.239,0.036,0.267,0.028,0.131,0.079,-0.052,0.05,"(1, 2)",20,0.5,85307,54169


### Per Tag Scores at the Winning Configuration


In [17]:
tfidf_tag_tables = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    best = sweep[sweep["split"] == name].sort_values("micro_f1", ascending=False).iloc[0]
    vectoriser = TfidfVectorizer(
        ngram_range=eval(best["ngram_range"]),
        min_df=int(best["min_df"]),
        max_df=float(best["max_df"]),
    )
    X_train = vectoriser.fit_transform(documents[parts["train"]])
    X_valid = vectoriser.transform(documents[parts["valid"]])
    y_train = target[parts["train"]]
    y_valid = target[parts["valid"]]
    scores = np.zeros((X_valid.shape[0], len(tag_vocabulary)))
    for j in range(len(tag_vocabulary)):
        model = MultinomialNB()
        model.fit(X_train, y_train[:, j])
        scores[:, j] = model.predict_proba(X_valid)[:, 1]

    predicted = (scores >= best["threshold"]).astype(np.int8)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_valid, predicted, average=None, zero_division=0,
    )
    tfidf_tag_tables[name] = pd.DataFrame({
        "tag": tag_vocabulary,
        "support": support,
        "predicted": predicted.sum(axis=0),
        "precision": precision.round(3),
        "recall": recall.round(3),
        "f1": f1.round(3),
    })
best_tags = tfidf_tag_tables["random"].sort_values("f1", ascending=False)

print("Best five tags at the winning configuration:")
display(best_tags.head(5))
print("Worst five tags at the winning configuration:")
display(best_tags.tail(5))
print("Tags above F1 0.3:", int((best_tags["f1"] > 0.3).sum()))
print("Tags below F1 0.1:", int((best_tags["f1"] < 0.1).sum()))
print("Tags never predicted:", int((best_tags["predicted"] == 0).sum()))

Best five tags at the winning configuration:


,tag,support,predicted,precision,recall,f1
16,hip-hop,594,484,0.663,0.540,0.596
19,rap,455,343,0.682,0.514,0.586
0,rock,3161,12058,0.262,0.998,0.415
8,metal,852,1062,0.348,0.434,0.387
1,pop,2362,9415,0.240,0.958,0.384


Worst five tags at the winning configuration:


,tag,support,predicted,precision,recall,f1
43,psychedelic,229,0,0.0,0.0,0.0
45,lo-fi,235,0,0.0,0.0,0.0
44,pop rock,200,0,0.0,0.0,0.0
46,fip,189,0,0.0,0.0,0.0
48,shoegaze,189,0,0.0,0.0,0.0


Tags above F1 0.3: 7
Tags below F1 0.1: 40
Tags never predicted: 9


### Compare the Per Tag Scores Against the Baseline


In [18]:
movement_rows = []
for name in ["random", "artist"]:
    reference = pd.read_csv(processed_dir / ("nb_bow_per_tag_" + name + ".csv"))
    merged = reference[["tag", "support", "f1"]].merge(
        tfidf_tag_tables[name][["tag", "f1"]], on="tag", suffixes=("_bow", "_tfidf"),
    )
    merged["change"] = (merged["f1_tfidf"] - merged["f1_bow"]).round(3)
    movement_rows.append(merged.assign(split=name))

    print(name, "split:",
          "tags improved", int((merged["change"] > 0).sum()),
          "tags worsened", int((merged["change"] < 0).sum()),
          "mean change", round(float(merged["change"].mean()), 3))
movement = pd.concat(movement_rows)
random_movement = movement[movement["split"] == "random"].sort_values("change")

print("Tags TF-IDF hurts most:")
display(random_movement.head(5))
print("Tags TF-IDF helps most:")
display(random_movement.tail(5))

random split: tags improved 4 tags worsened 46 mean change -0.068
artist split: tags improved 4 tags worsened 44 mean change -0.052
Tags TF-IDF hurts most:


,tag,support,f1_bow,f1_tfidf,change,split
18,rnb,449,0.234,0.004,-0.230,random
9,soul,663,0.259,0.042,-0.217,random
36,heavy metal,343,0.221,0.021,-0.200,random
15,dance,532,0.267,0.070,-0.197,random
32,metalcore,289,0.188,0.000,-0.188,random


Tags TF-IDF helps most:


,tag,support,f1_bow,f1_tfidf,change,split
45,lo-fi,235,0.006,0.000,-0.006,random
8,metal,852,0.373,0.387,0.014,random
19,rap,455,0.557,0.586,0.029,random
16,hip-hop,594,0.559,0.596,0.037,random
3,alternative,1636,0.214,0.257,0.043,random


### Save the Sweep Tables


In [19]:
sweep.to_csv(processed_dir / "tfidf_sweep.csv", index=False)

print("Wrote:", processed_dir / "tfidf_sweep.csv")

tfidf_comparison.to_csv(processed_dir / "tfidf_vs_bow.csv", index=False)

print("Wrote:", processed_dir / "tfidf_vs_bow.csv")

mass.to_csv(processed_dir / "tfidf_feature_mass.csv", index=False)

print("Wrote:", processed_dir / "tfidf_feature_mass.csv")

movement.to_csv(processed_dir / "tfidf_per_tag_change.csv", index=False)

print("Wrote:", processed_dir / "tfidf_per_tag_change.csv")

for name in ["random", "artist"]:
    out_path = processed_dir / ("tfidf_per_tag_" + name + ".csv")
    tfidf_tag_tables[name].to_csv(out_path, index=False)

    print(f"Wrote: {out_path}")

tfidf_comparison

Wrote: data\processed\tfidf_sweep.csv
Wrote: data\processed\tfidf_vs_bow.csv
Wrote: data\processed\tfidf_feature_mass.csv
Wrote: data\processed\tfidf_per_tag_change.csv
Wrote: data\processed\tfidf_per_tag_random.csv
Wrote: data\processed\tfidf_per_tag_artist.csv


,split,bow_micro_f1,tfidf_micro_f1_default_cut,tfidf_micro_f1_tuned_cut,micro_gain,bow_macro_f1,tfidf_macro_f1_tuned_cut,macro_gain,best_threshold,best_ngram_range,best_min_df,best_max_df,bow_terms,tfidf_terms
0,random,0.260,0.044,0.273,0.013,0.152,0.084,-0.068,0.05,"(1, 2)",20,0.5,86194,53828
1,artist,0.239,0.036,0.267,0.028,0.131,0.079,-0.052,0.05,"(1, 2)",20,0.5,85307,54169


## (expected) add logistic regression and linear SVM, compare against both references


### The Two References


In [20]:
references = []
for record in bow_summary.itertuples(index=False):
    references.append({
        "split": record.split,
        "reference": "naive bayes on counts",
        "micro_f1": record.micro_f1,
        "macro_f1": record.macro_f1,
    })
for record in tfidf_comparison.itertuples(index=False):
    references.append({
        "split": record.split,
        "reference": "naive bayes on tfidf",
        "micro_f1": record.tfidf_micro_f1_tuned_cut,
        "macro_f1": record.tfidf_macro_f1_tuned_cut,
    })
reference_table = pd.DataFrame(references)
best_reference = reference_table.loc[reference_table.groupby("split")["micro_f1"].idxmax()]

print("Winning TF-IDF settings carried over from the sweep:")
for record in tfidf_comparison.itertuples(index=False):
    print(f"  {record.split} ngram {record.best_ngram_range} min_df {record.best_min_df} max_df "
          f"{record.best_max_df} threshold {record.best_threshold}")
print()
print("Best reference micro F1 per split:",
      dict(zip(best_reference["split"], best_reference["micro_f1"])))
reference_table

Winning TF-IDF settings carried over from the sweep:
  random ngram (1, 2) min_df 20 max_df 0.5 threshold 0.05
  artist ngram (1, 2) min_df 20 max_df 0.5 threshold 0.05

Best reference micro F1 per split: {'artist': 0.267, 'random': 0.273}


,split,reference,micro_f1,macro_f1
0,random,naive bayes on counts,0.260,0.152
1,artist,naive bayes on counts,0.239,0.131
2,random,naive bayes on tfidf,0.273,0.084
3,artist,naive bayes on tfidf,0.267,0.079


### The Two Feature Sets


In [21]:
winning = tfidf_comparison.set_index("split")


def new_vectoriser(name, features):
    if features == "counts":
        return CountVectorizer()

    settings = winning.loc[name]

    return TfidfVectorizer(
        ngram_range=tuple(int(part) for part
                          in settings["best_ngram_range"].strip("()").split(",")),
        min_df=int(settings["best_min_df"]),
        max_df=float(settings["best_max_df"]),
    )


feature_sets = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    built = {}
    for label in ["counts", "tfidf"]:
        vectoriser = new_vectoriser(name, label)
        X_train = vectoriser.fit_transform(documents[parts["train"]])
        X_valid = vectoriser.transform(documents[parts["valid"]])
        built[label] = (X_train, X_valid)

        print(f"{name} {label} terms {X_train.shape[1]} mass per document "
              f"{round(float(X_train.sum(axis=1).mean()), 2)}")
    feature_sets[name] = built


random counts terms 86194 mass per document 230.02


random tfidf terms 53828 mass per document 8.89


artist counts terms 85307 mass per document 231.76


artist tfidf terms 54169 mass per document 8.9


### Train the Two Linear Models per Tag


In [22]:
import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

probability_cuts = [0.5, 0.3, 0.2, 0.15, 0.1, 0.05, 0.02]
margin_cuts = [0.0, -0.1, -0.25, -0.5, -0.75, -1.0]

model_rows = []
convergence_rows = []
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_train = target[parts["train"]]
    y_valid = target[parts["valid"]]
    for features in ["counts", "tfidf"]:
        X_train, X_valid = feature_sets[name][features]
        for learner in ["logistic", "svm"]:
            started = time.perf_counter()
            scores = np.zeros((X_valid.shape[0], len(tag_vocabulary)))
            not_converged = 0
            for j in range(len(tag_vocabulary)):
                if learner == "logistic":
                    model = LogisticRegression(max_iter=2000, random_state=seed)
                else:
                    model = LinearSVC(max_iter=2000, random_state=seed)
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always", ConvergenceWarning)
                    model.fit(X_train, y_train[:, j])
                    for item in caught:
                        if issubclass(item.category, ConvergenceWarning):
                            not_converged = not_converged + 1
                if learner == "logistic":
                    scores[:, j] = model.predict_proba(X_valid)[:, 1]
                else:
                    scores[:, j] = model.decision_function(X_valid)
            elapsed = time.perf_counter() - started
            convergence_rows.append({
                "split": name,
                "features": features,
                "model": learner,
                "tags_not_converged": not_converged,
                "seconds": round(elapsed, 1),
            })

            if learner == "logistic":
                cuts = probability_cuts
            else:
                cuts = margin_cuts

            for cut in cuts:
                predicted = (scores >= cut).astype(np.int8)
                micro = precision_recall_fscore_support(
                    y_valid, predicted, average="micro", zero_division=0,
                )
                macro = precision_recall_fscore_support(
                    y_valid, predicted, average="macro", zero_division=0,
                )
                model_rows.append({
                    "split": name,
                    "features": features,
                    "model": learner,
                    "cut": cut,
                    "default_cut": cut == cuts[0],
                    "micro_precision": round(micro[0], 3),
                    "micro_recall": round(micro[1], 3),
                    "micro_f1": round(micro[2], 3),
                    "macro_f1": round(macro[2], 3),
                    "predicted": int(predicted.sum()),
                })
            best_here = max(
                [r for r in model_rows
                 if r["split"] == name and r["features"] == features and r["model"] == learner],
                key=lambda r: r["micro_f1"],
            )

            print(name, features, learner,
                  "best micro F1", best_here["micro_f1"],
                  "at cut", best_here["cut"],
                  "tags not converged", not_converged,
                  "seconds", round(elapsed, 1))
linear = pd.DataFrame(model_rows)
convergence = pd.DataFrame(convergence_rows)

print()
print(f"Scored rows: {len(linear)}")

random counts logistic best micro F1 0.256 at cut 0.15 tags not converged 1 seconds 821.1


random counts svm best micro F1 0.216 at cut -0.5 tags not converged 50 seconds 740.5


random tfidf logistic best micro F1 0.32 at cut 0.15 tags not converged 0 seconds 33.0


random tfidf svm best micro F1 0.285 at cut -0.5 tags not converged 0 seconds 90.4


artist counts logistic best micro F1 0.235 at cut 0.15 tags not converged 3 seconds 810.8


artist counts svm best micro F1 0.198 at cut -0.75 tags not converged 50 seconds 740.0


artist tfidf logistic best micro F1 0.309 at cut 0.15 tags not converged 0 seconds 33.9


artist tfidf svm best micro F1 0.262 at cut -0.5 tags not converged 0 seconds 87.2

Scored rows: 52


### Convergence


In [23]:
print(convergence.to_string(index=False))

print()
print("Tags that failed to converge, by feature set:")
print(convergence.groupby("features")["tags_not_converged"].sum().to_string())

 split features    model  tags_not_converged  seconds
random   counts logistic                   1    821.1
random   counts      svm                  50    740.5
random    tfidf logistic                   0     33.0
random    tfidf      svm                   0     90.4
artist   counts logistic                   3    810.8
artist   counts      svm                  50    740.0
artist    tfidf logistic                   0     33.9
artist    tfidf      svm                   0     87.2

Tags that failed to converge, by feature set:
features
counts    104
tfidf       0


### What the Default Cuts Would Have Concluded


In [24]:
default_only = linear[linear["default_cut"]].copy()
tuned = linear.loc[linear.groupby(["split", "features", "model"])["micro_f1"].idxmax()].copy()

comparison = default_only.merge(
    tuned[["split", "features", "model", "cut", "micro_f1", "macro_f1"]],
    on=["split", "features", "model"],
    suffixes=("_default", "_tuned"),
)
columns = ["split", "features", "model", "micro_f1_default", "micro_f1_tuned",
           "macro_f1_default", "macro_f1_tuned", "cut_tuned"]
print(comparison[columns].to_string(index=False))

 split features    model  micro_f1_default  micro_f1_tuned  macro_f1_default  macro_f1_tuned  cut_tuned
random   counts logistic             0.200           0.256             0.151           0.192       0.15
random   counts      svm             0.198           0.216             0.160           0.171      -0.50
random    tfidf logistic             0.067           0.320             0.037           0.169       0.15
random    tfidf      svm             0.171           0.285             0.104           0.209      -0.50
artist   counts logistic             0.174           0.235             0.128           0.167       0.15
artist   counts      svm             0.173           0.198             0.132           0.149      -0.75
artist    tfidf logistic             0.063           0.309             0.037           0.149       0.15
artist    tfidf      svm             0.147           0.262             0.078           0.176      -0.50


### Compare Against Both References


In [25]:
reference_lookup = {}
for record in reference_table.itertuples(index=False):
    if record.reference == "naive bayes on counts":
        reference_lookup[(record.split, "counts")] = (record.micro_f1, record.macro_f1)
    else:
        reference_lookup[(record.split, "tfidf")] = (record.micro_f1, record.macro_f1)

against_rows = []
for record in tuned.itertuples(index=False):
    reference_micro, reference_macro = reference_lookup[(record.split, record.features)]
    against_rows.append({
        "split": record.split,
        "features": record.features,
        "model": record.model,
        "cut": record.cut,
        "micro_f1": record.micro_f1,
        "reference_micro_f1": reference_micro,
        "micro_gain": round(record.micro_f1 - reference_micro, 3),
        "macro_f1": record.macro_f1,
        "reference_macro_f1": reference_macro,
        "macro_gain": round(record.macro_f1 - reference_macro, 3),
    })
against = pd.DataFrame(against_rows).sort_values(["split", "features", "model"])

print(against.to_string(index=False))

print()
print("Beat their reference on micro F1:", int((against["micro_gain"] > 0).sum()), "of", len(against))
print("Beat their reference on macro F1:", int((against["macro_gain"] > 0).sum()), "of", len(against))

 split features    model   cut  micro_f1  reference_micro_f1  micro_gain  macro_f1  reference_macro_f1  macro_gain
artist   counts logistic  0.15     0.235               0.239      -0.004     0.167               0.131       0.036
artist   counts      svm -0.75     0.198               0.239      -0.041     0.149               0.131       0.018
artist    tfidf logistic  0.15     0.309               0.267       0.042     0.149               0.079       0.070
artist    tfidf      svm -0.50     0.262               0.267      -0.005     0.176               0.079       0.097
random   counts logistic  0.15     0.256               0.260      -0.004     0.192               0.152       0.040
random   counts      svm -0.50     0.216               0.260      -0.044     0.171               0.152       0.019
random    tfidf logistic  0.15     0.320               0.273       0.047     0.169               0.084       0.085
random    tfidf      svm -0.50     0.285               0.273       0.012     0.2

### The Best Model Overall


In [26]:
overall_rows = []
for name in ["random", "artist"]:
    here = tuned[tuned["split"] == name]
    winner = here.loc[here["micro_f1"].idxmax()]
    reference = best_reference[best_reference["split"] == name].iloc[0]
    overall_rows.append({
        "split": name,
        "best_model": winner["model"],
        "best_features": winner["features"],
        "best_cut": winner["cut"],
        "best_micro_f1": winner["micro_f1"],
        "best_macro_f1": winner["macro_f1"],
        "best_reference": reference["reference"],
        "reference_micro_f1": reference["micro_f1"],
        "reference_macro_f1": reference["macro_f1"],
        "micro_gain": round(winner["micro_f1"] - reference["micro_f1"], 3),
        "macro_gain": round(winner["macro_f1"] - reference["macro_f1"], 3),
    })
overall = pd.DataFrame(overall_rows)

print(overall.to_string(index=False))

 split best_model best_features  best_cut  best_micro_f1  best_macro_f1       best_reference  reference_micro_f1  reference_macro_f1  micro_gain  macro_gain
random   logistic         tfidf      0.15          0.320          0.169 naive bayes on tfidf               0.273               0.084       0.047       0.085
artist   logistic         tfidf      0.15          0.309          0.149 naive bayes on tfidf               0.267               0.079       0.042       0.070


### Per Tag Scores for the Winning Model


In [27]:
linear_tag_tables = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_train = target[parts["train"]]
    y_valid = target[parts["valid"]]
    winner = overall[overall["split"] == name].iloc[0]
    X_train, X_valid = feature_sets[name][winner["best_features"]]
    predicted = np.zeros((X_valid.shape[0], len(tag_vocabulary)), dtype=np.int8)
    for j in range(len(tag_vocabulary)):
        if winner["best_model"] == "logistic":
            model = LogisticRegression(max_iter=2000, random_state=seed)
        else:
            model = LinearSVC(max_iter=2000, random_state=seed)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            model.fit(X_train, y_train[:, j])
        if winner["best_model"] == "logistic":
            column = model.predict_proba(X_valid)[:, 1]
        else:
            column = model.decision_function(X_valid)
        predicted[:, j] = (column >= winner["best_cut"]).astype(np.int8)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_valid, predicted, average=None, zero_division=0,
    )
    linear_tag_tables[name] = pd.DataFrame({
        "tag": tag_vocabulary,
        "precision": precision.round(3),
        "recall": recall.round(3),
        "f1": f1.round(3),
        "support": support,
    }).sort_values("f1", ascending=False).reset_index(drop=True)

    print(name, "best five tags:", list(linear_tag_tables[name]["tag"].head(5)))
    print(name, "worst five tags:", list(linear_tag_tables[name]["tag"].tail(5)))
linear_tag_tables["random"].head(10)

random best five tags: ['rap', 'hip-hop', 'pop', 'rock', 'death metal']
random worst five tags: ['psychedelic', 'lo-fi', 'pop rock', 'fip', 'shoegaze']


artist best five tags: ['rap', 'hip-hop', 'rock', 'pop', 'death metal']
artist worst five tags: ['psychedelic', 'lo-fi', 'pop rock', 'fip', 'shoegaze']


,tag,precision,recall,f1,support
0,rap,0.684,0.666,0.675,455
1,hip-hop,0.701,0.650,0.674,594
2,pop,0.314,0.817,0.453,2362
3,rock,0.295,0.911,0.445,3161
4,death metal,0.408,0.473,0.438,273
5,metal,0.348,0.529,0.420,852
6,indie,0.266,0.773,0.396,2159
7,classic rock,0.318,0.300,0.309,733
8,female vocalists,0.218,0.435,0.290,1394
9,dance,0.294,0.271,0.282,532


### Where the Linear Models Help and Where They Do Not


In [28]:
change_rows = []
for name in ["random", "artist"]:
    bow_tags = bow_tag_tables[name]
    tfidf_tags = tfidf_tag_tables[name]
    merged = linear_tag_tables[name][["tag", "f1", "support"]].rename(columns={"f1": "f1_linear"})
    merged = merged.merge(bow_tags[["tag", "f1"]].rename(columns={"f1": "f1_bow"}), on="tag")
    merged = merged.merge(tfidf_tags[["tag", "f1"]].rename(columns={"f1": "f1_tfidf"}), on="tag")
    merged["change_vs_bow"] = (merged["f1_linear"] - merged["f1_bow"]).round(3)
    merged["change_vs_tfidf"] = (merged["f1_linear"] - merged["f1_tfidf"]).round(3)
    merged.insert(0, "split", name)
    change_rows.append(merged)

    for column, label in [("change_vs_bow", "counts"), ("change_vs_tfidf", "tfidf")]:
        print(name, "against naive bayes on", label + ":",
              "improved", int((merged[column] > 0).sum()),
              "worsened", int((merged[column] < 0).sum()),
              "level", int((merged[column] == 0).sum()),
              "mean change", round(float(merged[column].mean()), 3))
    print(name, "tags scored zero:",
          "winning model", int((merged["f1_linear"] == 0).sum()),
          "counts reference", int((merged["f1_bow"] == 0).sum()),
          "tfidf reference", int((merged["f1_tfidf"] == 0).sum()))
per_tag_change = pd.concat(change_rows, ignore_index=True)
biggest = per_tag_change.reindex(
    per_tag_change["change_vs_bow"].abs().sort_values(ascending=False).index
)

print()
print("Largest movements against the bag of words reference:")
print(biggest.head(10).to_string(index=False))

random against naive bayes on counts: improved 28 worsened 22 level 0 mean change 0.017
random against naive bayes on tfidf: improved 41 worsened 0 level 9 mean change 0.085
random tags scored zero: winning model 9 counts reference 0 tfidf reference 23
artist against naive bayes on counts: improved 29 worsened 19 level 2 mean change 0.018
artist against naive bayes on tfidf: improved 39 worsened 0 level 11 mean change 0.07
artist tags scored zero: winning model 11 counts reference 2 tfidf reference 24

Largest movements against the bag of words reference:
 split           tag  f1_linear  support  f1_bow  f1_tfidf  change_vs_bow  change_vs_tfidf
artist           rap      0.661      358   0.502     0.562          0.159            0.099
artist       hip-hop      0.623      458   0.480     0.516          0.143            0.107
random post-hardcore      0.210      372   0.079     0.000          0.131            0.210
random           rap      0.675      455   0.557     0.586          0.118 

### Save the Linear Model Tables


In [29]:
linear.to_csv(processed_dir / "linear_models.csv", index=False)
convergence.to_csv(processed_dir / "linear_convergence.csv", index=False)
against.to_csv(processed_dir / "linear_vs_references.csv", index=False)
overall.to_csv(processed_dir / "linear_best_overall.csv", index=False)
per_tag_change.to_csv(processed_dir / "linear_per_tag_change.csv", index=False)
for name in ["random", "artist"]:
    linear_tag_tables[name].to_csv(processed_dir / ("linear_per_tag_" + name + ".csv"), index=False)

print("Wrote: linear_models.csv")
print("Wrote: linear_convergence.csv")
print("Wrote: linear_vs_references.csv")
print("Wrote: linear_best_overall.csv")
print("Wrote: linear_per_tag_change.csv")
print("Wrote: linear_per_tag_random.csv")
print("Wrote: linear_per_tag_artist.csv")

Wrote: linear_models.csv
Wrote: linear_convergence.csv
Wrote: linear_vs_references.csv
Wrote: linear_best_overall.csv
Wrote: linear_per_tag_change.csv
Wrote: linear_per_tag_random.csv
Wrote: linear_per_tag_artist.csv


## (expected) time training and prediction at 10, 25, 50 and 100 percent of the data


### The Subsample Ladder


In [30]:
fractions = [0.10, 0.25, 0.50, 1.00]

ladders = {}
for name in ["random", "artist"]:
    train_rows = list(split_rows[name]["train"])
    rng = np.random.default_rng(seed)
    rng.shuffle(train_rows)
    ladder = {}
    for fraction in fractions:
        taken = int(round(fraction * len(train_rows)))
        ladder[fraction] = np.array(train_rows[:taken])
    ladders[name] = ladder

    print(name, "split, training rows at each size:",
          [len(ladder[fraction]) for fraction in fractions],
          "validation rows", len(split_rows[name]["valid"]))
for name in ["random", "artist"]:
    nested = True
    for smaller, larger in zip(fractions, fractions[1:]):
        if not set(ladders[name][smaller]).issubset(set(ladders[name][larger])):
            nested = False
    print(f"{name} split, each size is contained in the next: {nested}")

random split, training rows at each size: [5797, 14492, 28983, 57966] validation rows 12421
artist split, training rows at each size: [5800, 14499, 28998, 57997] validation rows 12424
random split, each size is contained in the next: True
artist split, each size is contained in the next: True


### What Gets Timed


In [31]:
learners = ["naive bayes", "logistic", "svm"]
timing_splits = ["random", "artist"]

repeat_limit = 3
repeat_budget = 60.0


def measure(work):
    best = None
    runs = 0
    while True:
        started = time.perf_counter()
        result = work()
        elapsed = time.perf_counter() - started
        runs = runs + 1
        if best is None or elapsed < best:
            best = elapsed
        if runs >= repeat_limit or best >= repeat_budget:
            break

    return result, best


print(f"Splits timed: {len(timing_splits)}")
print("Feature sets per split: 2")
print(f"Learners per feature set: {len(learners)}")
print(f"Sizes per learner: {len(fractions)}")
print(f"Models to train: "
      f"{len(timing_splits) * 2 * len(learners) * len(fractions) * len(tag_vocabulary)}")
print(f"A measurement under {int(repeat_budget)} seconds is run up to {repeat_limit} times and "
      f"the fastest is kept")


Splits timed: 2
Feature sets per split: 2
Learners per feature set: 3
Sizes per learner: 4
Models to train: 2400
A measurement under 60 seconds is run up to 3 times and the fastest is kept


### Run the Timing Grid


In [32]:
def new_model(learner):
    if learner == "naive bayes":
        return MultinomialNB()
    if learner == "logistic":
        return LogisticRegression(max_iter=2000, random_state=seed)
    return LinearSVC(max_iter=2000, random_state=seed)


def fit_pass(learner, X_train, y_train):
    models = []
    not_converged = 0
    for j in range(len(tag_vocabulary)):
        model = new_model(learner)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train, y_train[:, j])
            for item in caught:
                if issubclass(item.category, ConvergenceWarning):
                    not_converged = not_converged + 1
        models.append(model)
    return models, not_converged


def score_pass(learner, models, X_valid):
    scores = np.zeros((X_valid.shape[0], len(tag_vocabulary)))
    for j in range(len(tag_vocabulary)):
        if learner == "svm":
            scores[:, j] = models[j].decision_function(X_valid)
        else:
            scores[:, j] = models[j].predict_proba(X_valid)[:, 1]

    return scores


timing_rows = []
for name in timing_splits:
    parts = split_rows[name]
    y_valid = target[parts["valid"]]
    for fraction in fractions:
        rows = ladders[name][fraction]
        y_train = target[rows]
        for features in ["counts", "tfidf"]:
            vectoriser = new_vectoriser(name, features)
            X_train, vectorise_seconds = measure(
                lambda: vectoriser.fit_transform(documents[rows]))
            X_valid, transform_seconds = measure(
                lambda: vectoriser.transform(documents[parts["valid"]]))
            for learner in learners:
                fitted, fit_seconds = measure(
                    lambda: fit_pass(learner, X_train, y_train))
                models, not_converged = fitted
                scores, predict_seconds = measure(
                    lambda: score_pass(learner, models, X_valid))
                if learner == "svm":
                    cuts = margin_cuts
                else:
                    cuts = probability_cuts

                best_micro = -1.0
                best_macro = -1.0
                best_cut = None
                for cut in cuts:
                    predicted = (scores >= cut).astype(np.int8)
                    micro = precision_recall_fscore_support(
                        y_valid, predicted, average="micro", zero_division=0,
                    )[2]
                    macro = precision_recall_fscore_support(
                        y_valid, predicted, average="macro", zero_division=0,
                    )[2]
                    if micro > best_micro:
                        best_micro = micro
                        best_macro = macro
                        best_cut = cut
                timing_rows.append({
                    "split": name,
                    "fraction": fraction,
                    "train_rows": len(rows),
                    "valid_rows": len(parts["valid"]),
                    "features": features,
                    "model": learner,
                    "terms": X_train.shape[1],
                    "vectorise_seconds": round(vectorise_seconds, 3),
                    "fit_seconds": round(fit_seconds, 3),
                    "train_seconds": round(vectorise_seconds + fit_seconds, 3),
                    "transform_seconds": round(transform_seconds, 3),
                    "score_seconds": round(predict_seconds, 3),
                    "predict_seconds": round(transform_seconds + predict_seconds, 3),
                    "tags_not_converged": not_converged,
                    "best_cut": best_cut,
                    "micro_f1": round(float(best_micro), 3),
                    "macro_f1": round(float(best_macro), 3),
                })

                print(f"{name} {features} {learner} at {int(fraction * 100)} percent: train "
                      f"{round(vectorise_seconds + fit_seconds, 1)} s, predict "
                      f"{round(transform_seconds + predict_seconds, 2)} s, "
                      f"micro F1 {round(float(best_micro), 3)}")
timing = pd.DataFrame(timing_rows)

print()
print(f"Grid rows: {len(timing)}")


random counts naive bayes at 10 percent: train 0.6 s, predict 1.25 s, micro F1 0.197


random counts logistic at 10 percent: train 40.4 s, predict 1.1 s, micro F1 0.209


random counts svm at 10 percent: train 15.1 s, predict 1.08 s, micro F1 0.184


random tfidf naive bayes at 10 percent: train 1.1 s, predict 2.03 s, micro F1 0.254


random tfidf logistic at 10 percent: train 2.2 s, predict 1.88 s, micro F1 0.275


random tfidf svm at 10 percent: train 2.3 s, predict 1.87 s, micro F1 0.234


... (38 outputs left out)

artist counts svm at 100 percent: train 740.9 s, predict 1.04 s, micro F1 0.193


artist tfidf naive bayes at 100 percent: train 12.1 s, predict 2.29 s, micro F1 0.267


artist tfidf logistic at 100 percent: train 41.8 s, predict 2.1 s, micro F1 0.309


artist tfidf svm at 100 percent: train 100.8 s, predict 2.09 s, micro F1 0.262

Grid rows: 48


### Training Cost Against Data Size


In [33]:
size_columns = []
for fraction in fractions:
    size_columns.append("train_" + str(int(fraction * 100)))

scaling_rows = []
for name in timing_splits:
    for features in ["counts", "tfidf"]:
        for learner in learners:
            here = timing[(timing["split"] == name)
                          & (timing["features"] == features)
                          & (timing["model"] == learner)].sort_values("train_rows")
            slope = np.polyfit(np.log(here["train_rows"]), np.log(here["train_seconds"]), 1)[0]
            record = {
                "split": name,
                "features": features,
                "model": learner,
                "slope": round(float(slope), 2),
            }
            for fraction, column in zip(fractions, size_columns):
                record[column] = here[here["fraction"] == fraction]["train_seconds"].iloc[0]
            record["growth"] = round(
                float(record[size_columns[-1]] / record[size_columns[0]]), 1,
            )
            scaling_rows.append(record)
scaling = pd.DataFrame(scaling_rows)

columns = ["split", "features", "model"] + size_columns + ["growth", "slope"]

print("Training seconds at each size, and how they grow:")
print(scaling[columns].to_string(index=False))

rising = 0
for record in scaling.itertuples(index=False):
    seconds = [getattr(record, column) for column in size_columns]
    if all(later > earlier for earlier, later in zip(seconds, seconds[1:])):
        rising = rising + 1

print()
print(f"Combinations whose seconds rise at every step: {rising} of {len(scaling)}")

Training seconds at each size, and how they grow:
 split features       model  train_10  train_25  train_50  train_100  growth  slope
random   counts naive bayes     0.598     1.354     2.639      5.343     8.9   0.95
random   counts    logistic    40.394   131.815   330.575    840.729    20.8   1.32
random   counts         svm    15.120    70.197   234.657    754.211    49.9   1.70
random    tfidf naive bayes     1.145     2.901     5.993     12.148    10.6   1.03
random    tfidf    logistic     2.244     8.918    20.181     43.068    19.2   1.28
random    tfidf         svm     2.256     7.957    18.389    101.268    44.9   1.60
artist   counts naive bayes     0.561     1.356     2.644      5.254     9.4   0.97
artist   counts    logistic    41.147   133.284   329.633    841.831    20.5   1.31
artist   counts         svm    15.989    77.858   232.274    740.872    46.3   1.66
artist    tfidf naive bayes     1.154     2.914     5.989     12.142    10.5   1.02
artist    tfidf    logisti

### Prediction Cost Against Data Size


In [34]:
columns = ["split", "features", "model", "terms", "transform_seconds",
           "score_seconds", "predict_seconds"]
for name in timing_splits:
    here = timing[(timing["split"] == name) & (timing["fraction"].isin([0.10, 1.00]))]
    print(f"{name} split, prediction at the smallest and the largest size:")
    print(here.sort_values(["features", "model", "fraction"])[columns].to_string(index=False))
    print()

ratio = timing.groupby(["features", "model"])["predict_seconds"].agg(["min", "max"])
ratio["spread"] = (ratio["max"] / ratio["min"]).round(1)

print("Prediction seconds across the whole grid, by feature set and learner:")
print(ratio.round(2).to_string())

random split, prediction at the smallest and the largest size:
 split features       model  terms  transform_seconds  score_seconds  predict_seconds
random   counts    logistic  27361              0.927          0.173            1.100
random   counts    logistic  86194              0.927          0.184            1.111
random   counts naive bayes  27361              0.927          0.320            1.247
random   counts naive bayes  86194              0.927          0.378            1.305
random   counts         svm  27361              0.927          0.148            1.075
random   counts         svm  86194              0.927          0.164            1.092
random    tfidf    logistic   6887              1.777          0.108            1.885
random    tfidf    logistic  53828              2.042          0.167            2.208
random    tfidf naive bayes   6887              1.777          0.248            2.025
random    tfidf naive bayes  53828              2.042          0.350         

### Save the Timing Tables


In [35]:
timing.to_csv(processed_dir / "timing_study.csv", index=False)
scaling.to_csv(processed_dir / "timing_scaling.csv", index=False)

print("Wrote: timing_study.csv")
print("Wrote: timing_scaling.csv")

Wrote: timing_study.csv
Wrote: timing_scaling.csv
